# MTLFuseNet — Training + LOSO Cross-Validation

End-to-end training and **Leave-One-Subject-Out (LOSO)** cross-validation for MTLFuseNet on DREAMER.
Calls the reusable modules (`mtl_preprocess.py`, `mtl_model.py`, `mtl_loso.py`).

**Paper targets (DREAMER, subject-independent LOSO):** valence **80.43%**, arousal **83.33%**.

> Run on a **GPU** runtime (Runtime → Change runtime type → GPU).

## 1. Environment setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install eegproc
# some runtimes ship a scipy wheel whose binary fails to load — pin fresh
!pip -q install --upgrade 'scipy>=1.16' 'numpy>=2.2' scikit-learn

In [ ]:
# push the mtl-fusenet branch, then clone it here (contains models.py, losses.py,
# preprocessing.py + the new mtl_*.py modules)
!git clone -b mtl-fusenet https://github.com/VitorInserra/EEGProc.git 2>/dev/null || (cd EEGProc && git pull)
%cd EEGProc
# --- OR upload these 6 files instead of cloning: ---
# models.py losses.py preprocessing.py mtl_preprocess.py mtl_model.py mtl_loso.py

In [ ]:
import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Paths
Point at your DREAMER CSV on Drive; cache features to Drive so preprocessing runs **once**.

In [ ]:
CSV       = '/content/drive/MyDrive/EEGProc/dreamer_joined.csv'   # <-- edit
PROCESSED = '/content/drive/MyDrive/EEGProc/processed_trials'      # <-- edit

## 3. Preprocess + cache (run once)
Filter → differential-entropy features → mutual-information adjacency (3 bands θ/α/β) → 9×9 grid
windows, for all 23×18 trials. Skip if the cache already exists.

In [ ]:
import os
from mtl_preprocess import preprocess_all
if not os.path.exists(os.path.join(PROCESSED, 'manifest.pkl')):
    manifest, errors = preprocess_all(CSV, out_dir=PROCESSED, mi_max_samples=5000)
    print(f'cached {len(manifest)} trials, {len(errors)} errors')
else:
    print('cache found — skipping preprocessing')

## 4. LOSO — Valence
23 folds. `lr=1e-4` and dropout 0.2 follow the paper's Table 2; `epochs` is not specified in the
paper — tune it (start ~50-100).

In [ ]:
from mtl_loso import run_loso
res_v = run_loso(PROCESSED, task='valence', epochs=50, batch_size=64, lr=1e-4)
v = res_v['dreamer_valence']['mean_scores']
print(f"VALENCE  acc={v['accuracy']:.4f}  f1={v['f1']:.4f}   (paper: 0.8043)")

## 5. LOSO — Arousal

In [ ]:
res_a = run_loso(PROCESSED, task='arousal', epochs=50, batch_size=64, lr=1e-4)
a = res_a['dreamer_arousal']['mean_scores']
print(f"AROUSAL  acc={a['accuracy']:.4f}  f1={a['f1']:.4f}   (paper: 0.8333)")

## 6. Per-subject results

In [ ]:
import matplotlib.pyplot as plt, numpy as np
def plot_task(res, task, paper):
    um = res[f'dreamer_{task}']['user_metrics']
    subs=[u['subject_id'] for u in um]; accs=[u['accuracy'] for u in um]
    plt.bar(range(len(subs)), accs)
    plt.axhline(np.mean(accs), ls='--', color='k', label=f'mean={np.mean(accs):.3f}')
    plt.axhline(paper, ls=':', color='r', label=f'paper={paper}')
    plt.xticks(range(len(subs)), subs); plt.xlabel('held-out subject'); plt.ylabel('accuracy')
    plt.title(f'LOSO per-subject accuracy — {task}'); plt.legend(); plt.show()
plot_task(res_v, 'valence', 0.8043)
plot_task(res_a, 'arousal', 0.8333)

---
### Reconciliation with the paper (Li et al., KBS 2023)
**Matches Table 2 (DREAMER):** 3 bands θ/α/β (gamma unavailable — data is 4–30 Hz), VAE 4 conv
layers 128/256/256/512, spatio-temporal latent 128, spatio-spectral (GRU) 384, fusion→512, 14×14 MI
adjacency + symmetric norm, loss weights 0.7/0.2/0.1, label median 3, LOSO 22-train/1-test.
Applied from Table 2: **lr 1e-4**, **dropout 0.2**.

**Not specified in the paper — confirm / tune:** epochs (MaxIter), batch size, optimizer (Adam
assumed), focal-loss α & γ and triplet margin (defaults 0.7 / 2.0 / 1.0 in `mtl_model.py`).

**Open modeling choice:** each trial currently uses *both* baseline + stimulus EEG. The paper
describes stimulus-video watching (65–393 s) — consider stimulus-only in
`mtl_preprocess.build_trial_sample`. Also: paper's focal loss (Eq. 20) weights the negative class by
(1−α); `losses.py` uses α for both — a minor alignment to consider.